# 05. Subagents, skills и memory

Агент получает долговременные инструкции (`AGENTS.md`), локальные skills
и двух subagents: исследователь репозитория и code reviewer.

Connectors теперь подключаются через единый `CONNECTOR_TOOLS` из пакета `connectors`.

Генерируется `agents/step_05_full.py`.

```bash
uv run langgraph dev --config langgraph.steps.json
```

## Визуальная рамка

![Subagents, skills and memory](visuals/05_subagents_memory.svg)

Этот шаг связывает материалы воркшопа со слайдом про кукбуки: agent runtime становится расширяемым через local rules, skills и специализированных subagents.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend, LocalShellBackend
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'pyproject.toml').exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')

from connectors import CONNECTOR_TOOLS

### Subagents

Два встроенных subagent: `repo-researcher` (исследует код) и `reviewer` (ревьюит патчи).

In [ ]:
SUBAGENTS = [
    {
        "name": "repo-researcher",
        "description": "Map repository structure, APIs, tests, and likely change locations.",
        "system_prompt": (
            "You research codebases. Inspect files, identify relevant modules, "
            "and return concise findings with file paths and rationale."
        ),
    },
    {
        "name": "reviewer",
        "description": "Review proposed patches for bugs, missing tests, and regressions.",
        "system_prompt": (
            "You are a code reviewer. Focus on correctness, edge cases, tests, "
            "security, and maintainability. Be specific and concise."
        ),
    },
]

### Собираем агента

In [ ]:
DEFAULT_MODEL = 'openrouter:tencent/hy3:free'

def model_name() -> str:
    return os.getenv('OPENCLAW_MODEL', DEFAULT_MODEL)


def workspace_root() -> Path:
    return Path(os.getenv('OPENCLAW_WORKSPACE', '.')).expanduser().resolve()


def backend():
    root_dir = workspace_root()
    if os.getenv('OPENCLAW_ENABLE_LOCAL_SHELL') != '1':
        return FilesystemBackend(root_dir=root_dir, virtual_mode=True)
    return LocalShellBackend(
        root_dir=root_dir, virtual_mode=True,
        inherit_env=True, timeout=120, max_output_bytes=80_000,
    )


BASE_PROMPT = """\
You are OpenClaw, an experimental open-source coding agent built with LangChain
Deep Agents. You help with software engineering, repository navigation, product
research, and implementation.

Work like a careful senior engineer:
- inspect before editing;
- keep changes focused;
- verify with tests or equivalent checks;
- explain only the useful result to the user.
"""


agent = create_deep_agent(
    model=model_name(),
    tools=CONNECTOR_TOOLS,
    system_prompt=BASE_PROMPT,
    subagents=SUBAGENTS,
    skills=["/skills/swe-resolution", "/skills/product-research"],
    memory=["/AGENTS.md"],
    backend=backend(),
    interrupt_on={"execute": True},
)

print(type(agent).__name__)
print(f"Tools: {len(CONNECTOR_TOOLS)}")
print(f"Subagents: {len(SUBAGENTS)}")

### Сгенерировать entrypoint

In [ ]:
import json

AGENTS_DIR = REPO_ROOT / "agents"
AGENTS_DIR.mkdir(exist_ok=True)

agent_code = '''"""Step 05: subagents, skills, memory, and all connectors."""

from __future__ import annotations

import os
from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend, LocalShellBackend
from dotenv import load_dotenv

from connectors import CONNECTOR_TOOLS

load_dotenv()

DEFAULT_MODEL = "openrouter:tencent/hy3:free"


def _workspace_root() -> Path:
    return Path(os.getenv("OPENCLAW_WORKSPACE", ".")).expanduser().resolve()


def _backend():
    root_dir = _workspace_root()
    if os.getenv("OPENCLAW_ENABLE_LOCAL_SHELL") != "1":
        return FilesystemBackend(root_dir=root_dir, virtual_mode=True)
    return LocalShellBackend(
        root_dir=root_dir, virtual_mode=True,
        inherit_env=True, timeout=120, max_output_bytes=80_000,
    )


SUBAGENTS = [
    {"name": "repo-researcher", "description": "Map repository structure, APIs, tests, and likely change locations.", "system_prompt": "You research codebases. Inspect files, identify relevant modules, and return concise findings with file paths and rationale."},
    {"name": "reviewer", "description": "Review proposed patches for bugs, missing tests, and regressions.", "system_prompt": "You are a code reviewer. Focus on correctness, edge cases, tests, security, and maintainability. Be specific and concise."},
]


agent = create_deep_agent(
    model=os.getenv("OPENCLAW_MODEL", DEFAULT_MODEL),
    tools=CONNECTOR_TOOLS,
    system_prompt="""\
You are OpenClaw, an experimental open-source coding agent built with LangChain
Deep Agents. You help with software engineering, repository navigation, product
research, and implementation.

Work like a careful senior engineer:
- inspect before editing;
- keep changes focused;
- verify with tests or equivalent checks;
- explain only the useful result to the user.
""",
    subagents=SUBAGENTS,
    skills=["/skills/swe-resolution", "/skills/product-research"],
    memory=["/AGENTS.md"],
    backend=_backend(),
    interrupt_on={"execute": True},
)
'''

step_file = AGENTS_DIR / "step_05_full.py"
step_file.write_text(agent_code)

steps_config = {
    "dependencies": ["."],
    "graphs": {
        "openclaw": "./agents/step_05_full.py:agent",
    },
    "env": ".env",
}
config_path = REPO_ROOT / "langgraph.steps.json"
config_path.write_text(json.dumps(steps_config, indent=2) + "\n")

print(f"Wrote {step_file}")
print(f"Wrote {config_path}")
print()
print("Restart:")
print("  uv run langgraph dev --config langgraph.steps.json")

### Проверочные prompts

После перезапуска попробуй:

> "Inspect the workspace files and summarize what this project does."

> "Use the Telegram connector in dry-run mode. Prepare a short workshop message."